In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [16]:
!nvidia-smi

Tue Mar 10 01:20:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   43C    P8              1W /  120W |     145MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
# 1. Device Setup. 
# This is standard boilerplate. It checks if an NVIDIA GPU is available.
# If yes, it uses "cuda". If no, it falls back to "cpu".
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [18]:
# 2. Let's create a dummy dataset: 1000 samples, 20 features each.
# We wrap our raw tensors in a TensorDataset, which pairs inputs with their targets.
X_data = torch.randn(1000, 20)
y_data = torch.randn(1000, 1)
dataset = TensorDataset(X_data, y_data)

In [19]:
# 3. Initialize the DataLoader.
# batch_size=32 means the model will process 32 rows at a time.
# shuffle=True ensures the order is randomized every epoch.
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [20]:
# 4. Initialize model and MOVE IT TO THE GPU.
model = nn.Linear(20, 1)
model.to(device)  # CRITICAL: This copies the model's weights into GPU VRAM.

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

epochs = 5
for epoch in range(epochs):
    # Instead of doing one massive forward pass, we loop through the mini-batches
    for batch_X, batch_y in dataloader:
        
        # CRITICAL PITFALL: Device Mismatch
        # The DataLoader loads data into system RAM (CPU). 
        # Our model is on the GPU. We MUST move the data to the GPU before the forward pass.
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        # The standard 5-step loop we learned in Checkpoint 1.3
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
    print(f"Epoch {epoch+1} Complete. Final Batch Loss: {loss.item():.4f}")

Epoch 1 Complete. Final Batch Loss: 0.5956
Epoch 2 Complete. Final Batch Loss: 0.4120
Epoch 3 Complete. Final Batch Loss: 1.3148
Epoch 4 Complete. Final Batch Loss: 1.1590
Epoch 5 Complete. Final Batch Loss: 0.9198


# Excersise 1.3

In [21]:
# Setingup Device variable
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [22]:
#Dummy data
x_data = torch.rand(500, 4)
y_data = torch.rand(500, 1)
dataset = TensorDataset(x_data, y_data)

In [23]:
dataloader = DataLoader(dataset, batch_size = 16, shuffle=True)
model = nn.Linear(4, 1)
model.to(device)

Linear(in_features=4, out_features=1, bias=True)

In [24]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [28]:
epochs = 5
for epoch in range(epochs):
    for batch_x, batch_y in dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        #5 things
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} Complete. Final Batch Loss: {loss.item():.4f}")        

Epoch 1 Complete. Final Batch Loss: 0.1542
Epoch 2 Complete. Final Batch Loss: 0.0943
Epoch 3 Complete. Final Batch Loss: 0.0096
Epoch 4 Complete. Final Batch Loss: 0.0922
Epoch 5 Complete. Final Batch Loss: 0.1958
